In [1]:
import glob
import h5py
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 100
import numpy as np
from astropy.io import fits
from astropy.wcs import WCS
from tqdm import tqdm
from pandas import DataFrame


In [218]:
def plot_map(map, wcs=None, lowp=0.1, highp=95, fig=None, ax=None, colorbar=True, low=None, high=None, cmap='viridis', norm=LogNorm,
            cbar_label='MJy/sr'):
    if low is None or high is None:
        high, low = np.nanpercentile(map[map>0], [highp, lowp])
    if fig is None:
        fig = plt.figure(figsize=(10, 10))
    if ax is None:
        ax = fig.add_subplot(111, projection=wcs)
    if norm is LogNorm:
        im = ax.imshow(map, norm=LogNorm(vmin=low, vmax=high), origin='lower', cmap=cmap)
    else:
        im = ax.imshow(map, vmin=low, vmax=high, origin='lower', cmap=cmap)

    if wcs is not None:
        # Explicitly set axis labels
        ax.coords['ra'].set_axislabel('RA')
        ax.coords['ra'].set_axislabel_position('b')  # Ensure RA label is only at the bottom
        ax.coords['ra'].set_ticks_position('b')  # Set RA ticks only at the bottom
        ax.coords['ra'].set_ticklabel_position('b')  # Set RA tick labels only at the bottom
        ax.coords['dec'].set_axislabel('DEC')
        ax.grid(color='black', linestyle='--', alpha=0.5)

    if colorbar:
        cbar = plt.colorbar(im, ax=ax, orientation='vertical', fraction=0.040, pad=0.04)
        cbar.set_label(cbar_label)
    return fig, ax, im

# Euclid L3 Mosaic

In [ ]:
mosaic_list = sorted(glob.glob('/data1/Euclid/mosaics/NIR_Y/*.fits'))

In [ ]:
# Check center pixel position of each mosaic
center_positions = []
for mosaic_file in tqdm(mosaic_list):
    with fits.open(mosaic_file) as hdul:
        wcs = WCS(hdul[0].header)
        ny, nx = hdul[0].data.shape
        center_x, center_y = nx // 2, ny // 2
        ra_dec = wcs.wcs_pix2world([[center_x, center_y]], 0)[0]
        center_positions.append(ra_dec)

df_mosaic = DataFrame(mosaic_list, columns=['file_path'])
df_mosaic['center_ra'] = [pos[0] for pos in center_positions]
df_mosaic['center_dec'] = [pos[1] for pos in center_positions]

In [ ]:
nep_ra, nep_dec = 270, 66.5
df_mosaic['sep_from_nep'] = np.sqrt((df_mosaic['center_ra'] - nep_ra)**2 + (df_mosaic['center_dec'] - nep_dec)**2)
df_mosaic.sort_values('sep_from_nep', inplace=True)

hdul_mer = fits.open(df_mosaic.iloc[0]['file_path'])
data_mer = hdul_mer[0].data
wcs_mer = WCS(hdul_mer[0].header)

In [ ]:
plot_map(data_mer, wcs=wcs_mer, low=-0.5, high=2, colorbar=False, cmap='viridis', norm=None, cbar_label='$e^-$')
plt.title('MER Mosaic')
plt.savefig('/home/thomasli/spherex/selfcal/figures/mer_mosaic.png', dpi=300)
plt.close()

# Euclid SelfCal Mosaic

In [ ]:
hdul_sc = fits.open('/data3/thomasli/selfcal/old_euclid_outputs/EDFN_Y_0p6arcsec_coadd.fits')

In [ ]:
data_sc = hdul_sc[0].data
wcs_sc = WCS(hdul_sc[0].header)

In [ ]:
from reproject import reproject_interp

In [ ]:
data_sc_reproj, _ = reproject_interp((data_sc, wcs_sc), wcs_mer, shape_out=data_mer.shape, parallel=20)

In [ ]:
mer50, mer95 = np.nanpercentile(data_mer, [70, 99])
sc50, sc95 = np.nanpercentile(data_sc_reproj, [70, 99])


In [ ]:
data_compare = (data_sc_reproj - sc50) * (mer95 - mer50) / (sc95 - sc50) + mer50

In [ ]:
np.nanpercentile(data_compare, [50, 99])

In [ ]:
exp_dir = '/data1/Euclid/OTF/NISP/'
exposure_list = glob.glob(exp_dir + '*_Y*.fits')

In [ ]:
# fits.open(exposure_list[0])[1].header

In [204]:
fig, ax = plt.subplots(1, 2, figsize=(12, 6), subplot_kw={'projection': wcs_mer})
plt.subplot(1,2,1)
plot_map(data_mer, wcs=wcs_mer, low=-0.5, high=1.5, colorbar=False, cmap='viridis', norm=None, cbar_label='$e^-$', fig=fig, ax=ax[0])
plt.grid(color='white', alpha=0.5)
plt.title('MER Mosaic')
plt.subplot(1,2,2)
plot_map(data_compare, wcs=wcs_mer, low=-0.5, high=2, colorbar=False, cmap='viridis', norm=None, cbar_label='$e^-$', fig=fig, ax=ax[1])
plt.grid(color='white', alpha=0.5)
plt.title('Self-Cal Mosaic')
plt.savefig('/home/thomasli/spherex/selfcal/figures/compare_mosaic.png', dpi=200, bbox_inches='tight', transparent=True)
plt.close()

# Euclid Exposures

In [205]:
exp_dir = '/data1/Euclid/OTF/NISP/'
exposure_list = glob.glob(exp_dir + '*_Y*.fits')

In [206]:
hdul = fits.open(exposure_list[0])

In [207]:
fig, axs = plt.subplots(4, 4, figsize=(12, 12))

for i in range(16):
    ax = axs[i//4, i%4]
    data = hdul[i*3 + 1].data 
    im = ax.imshow(data, vmin=50, vmax=90, origin='lower', cmap='viridis')
    ax.set_title(f'Detector {i+1}')
    ax.axis('off')
    ax.set_aspect('equal') # Ensures pixels remain square

fig.subplots_adjust(wspace=0.05, hspace=0, right=0.8)
cbar_ax = fig.add_axes([0.81, 0.15, 0.02, 0.7]) 

fig.colorbar(im, cax=cbar_ax, label='Pixel Value [$e^-$]')

plt.savefig('/home/thomasli/spherex/selfcal/figures/euclid_l2.png', dpi=200, bbox_inches='tight', transparent=True)
plt.close()

# SPHEREx Exposures

In [208]:
exp_list = [f'/mnt/md127/SPHEREx/reproc_data/deep_north/2025W30_2B/l2b-v14-2025-210/{i}/level2_2025W30_2B_0650_4D{i}_spx_l2b-v14-2025-210.fits'
            for i in range(1, 7)]

In [209]:
data = []
for f in exp_list:
    with fits.open(f) as hdul:
        data.append(hdul[1].data)

In [210]:
fig, axs = plt.subplots(2, 3, figsize=(12, 7))
for i in range(6):
    ax = axs[i//3, i%3]
    # vmin, vmax = np.nanpercentile(data[i], [1, 90])
    im = ax.imshow(data[i], vmin=0.03, vmax=0.4, origin='lower')
    ax.set_title(f'Detector {i+1}')
    ax.axis('off')  # no axis for clarity
    ax.set_aspect('equal') # Ensures pixels remain square

fig.subplots_adjust(wspace=0.1, hspace=0.1, right=0.8)
cbar_ax = fig.add_axes([0.81, 0.15, 0.02, 0.7]) 

fig.colorbar(im, cax=cbar_ax, label='Pixel Value [MJy/sr]')

plt.savefig('/home/thomasli/spherex/selfcal/figures/spherex_l2.png', dpi=200, bbox_inches='tight', transparent=True)
plt.close()

# SPHEREx Wavelength

In [211]:
from SelfCal.SPHERExUtility import load_calibration

In [212]:
wav_data = []

for i in range(1, 7):
    det_BC, det_BW = load_calibration(band=i, calibration_dir='/home/thomasli/spherex/SPHEREx_Spectral_Calibration')
    wav_data.append(det_BC)

In [213]:
fig, axs = plt.subplots(2, 3, figsize=(12, 7))
for i in range(6):
    ax = axs[i//3, i%3]
    im = ax.imshow(wav_data[i], vmin=0.7, vmax=5.0, origin='lower')
    ax.set_title(f'Detector {i+1}')
    ax.axis('off')  # no axis for clarity
    ax.set_aspect('equal') # Ensures pixels remain square

fig.subplots_adjust(wspace=0.1, hspace=0.1, right=0.8)
cbar_ax = fig.add_axes([0.81, 0.15, 0.02, 0.7]) 

fig.colorbar(im, cax=cbar_ax, label='Wavelength [$\\mu m$]')

plt.savefig('/home/thomasli/spherex/selfcal/figures/spherex_wav.png', dpi=200, bbox_inches='tight', transparent=True)
plt.close()

# Euclid Offsets

In [12]:
cal_file = '/data3/thomasli/selfcal/old_euclid_outputs/nep_full_H_3arcsec_40chunk_fit.h5'
with h5py.File(cal_file, 'r') as hf:
    D = hf['D'][:]

D -= D.min()

In [13]:
from scipy.interpolate import RectBivariateSpline
def mean_preserving_spline_2d(x_edges, y_edges, z_means, kx=3, ky=3):
    """
    Generates a mean-preserving spline function f(x, y) based on 2D grid edges
    and the average value z_mean in each rectangular cell.

    The function f(x, y) is constructed as the mixed partial derivative of a 
    bicubic spline F(x, y), where F(x, y) is the double integral of f(x, y).

    Parameters
    ----------
    x_edges : array_like, shape (N+1,)
        The edges of the bins along the x-axis.
    y_edges : array_like, shape (M+1,)
        The edges of the bins along the y-axis.
    z_means : array_like, shape (N, M)
        The mean value of the function within each rectangular bin.
    kx, ky : int, optional
        The degrees of the spline in x and y directions. Default is 3 (bicubic).
        
    Returns
    -------
    f_spline : callable
        A function f(x, y, grid=False) that evaluates the interpolated density.
        - If grid=True, evaluates on the grid spanned by x and y vectors.
        - If grid=False, evaluates at coordinates (x, y).
    """
    x_edges = np.asarray(x_edges, dtype=float)
    y_edges = np.asarray(y_edges, dtype=float)
    z_means = np.asarray(z_means, dtype=float)

    # 1. Validation
    if x_edges.shape[0] != z_means.shape[0] + 1:
        raise ValueError("len(x_edges) must be len(z_means) + 1 (rows)")
    if y_edges.shape[0] != z_means.shape[1] + 1:
        raise ValueError("len(y_edges) must be z_means.shape[1] + 1 (cols)")

    # 2. Compute Interval Areas (Volumes)
    # dx: (N,) -> column vector (N, 1)
    # dy: (M,) -> row vector (1, M)
    dx = np.diff(x_edges)
    dy = np.diff(y_edges)
    
    # Calculate volume of each cell: z_mean * dx * dy
    # Broadcasting: (N, 1) * (1, M) * (N, M)
    cell_volumes = z_means * dx[:, None] * dy[None, :]

    # 3. Compute Cumulative Integrals (2D Cumulative Sum)
    # The integral at (x0, y0) is 0.
    # The integral at (xi, yj) is sum of volumes in rect defined by [0:i, 0:j]
    integral_grid = np.zeros((len(x_edges), len(y_edges)))
    
    # cumsum over x (axis 0), then over y (axis 1)
    cumulative_volumes = np.cumsum(np.cumsum(cell_volumes, axis=0), axis=1)
    
    # Fill the integral grid (row 0 and col 0 remain 0s)
    integral_grid[1:, 1:] = cumulative_volumes

    # 4. Interpolate the Integral Surface F(x, y)
    # RectBivariateSpline creates a smooth surface over the regular grid
    F_spline = RectBivariateSpline(x_edges, y_edges, integral_grid, kx=kx, ky=ky)

    # 5. Define the Density Function
    # f(x, y) = d^2 F / dx dy
    # RectBivariateSpline allows calculating partial derivatives directly.
    
    def f_spline(x, y, grid=False):
        """
        Evaluate the mean-preserving spline.
        grid=True: x and y are vectors defining a grid.
        grid=False: x and y are coordinates of points.
        """
        # Calculate mixed partial derivative (dx=1, dy=1)
        return F_spline(x, y, dx=1, dy=1, grid=grid)

    return f_spline

In [14]:
def interpolate_offset(offset, n_chunk):
    x_edges = np.linspace(0, 2040, n_chunk+1)
    y_edges = np.linspace(0, 2040, n_chunk+1)
    spl = mean_preserving_spline_2d(x_edges, y_edges, offset, kx=2, ky=2)


    x_fine_edges = np.linspace(0, 2040, 2040+1)
    y_fine_edges = np.linspace(0, 2040, 2040+1)
    x_med, y_med = x_fine_edges[:-1] + (x_fine_edges[1] - x_fine_edges[0])/2, y_fine_edges[:-1] + (y_fine_edges[1] - y_fine_edges[0])/2

    fine_offset = spl(x_med, y_med, grid=True)
    return fine_offset

In [15]:
interpolated_offsets = []
for O in D:
    interpolated_offsets.append(interpolate_offset(O, n_chunk=40))

In [18]:
fig, axs = plt.subplots(4, 4, figsize=(12, 12))

for i in range(16):
    ax = axs[i//4, i%4]
    data = interpolated_offsets[i]
    im = ax.imshow(data, vmin=0, vmax=8, origin='lower', cmap='viridis')
    ax.set_title(f'Detector {i+1}')
    ax.axis('off')
    ax.set_aspect('equal') # Ensures pixels remain square

fig.subplots_adjust(wspace=0.05, hspace=0, right=0.8)
cbar_ax = fig.add_axes([0.81, 0.15, 0.02, 0.7]) 

fig.colorbar(im, cax=cbar_ax, label='Pixel Value [$e^-$]')

plt.savefig('/home/thomasli/spherex/selfcal/figures/euclid_offset.png', dpi=200, bbox_inches='tight', transparent=True)
plt.close()

# Euclid Mosaics

In [214]:
wcs = WCS(fits.open('/data3/thomasli/selfcal/old_euclid_outputs/EDFN_Y_0p6arcsec_coadd.fits')[0].header)

In [215]:
y_mos = fits.open('/data3/thomasli/selfcal/old_euclid_outputs/EDFN_Y_0p6arcsec_coadd.fits')[0].data
j_mos = fits.open('/data3/thomasli/selfcal/old_euclid_outputs/EDFN_J_0p6arcsec_coadd.fits')[0].data
h_mos = fits.open('/data3/thomasli/selfcal/old_euclid_outputs/EDFN_H_0p6arcsec_coadd.fits')[0].data

y_mos -= np.percentile(y_mos[y_mos>0], 0.1)
j_mos -= np.percentile(j_mos[j_mos>0], 0.1)
h_mos -= np.percentile(h_mos[h_mos>0], 0.1)

y_mos[y_mos<0] = np.nan
j_mos[j_mos<0] = np.nan
h_mos[h_mos<0] = np.nan

In [219]:
shared_norm = LogNorm(vmin=0.001, vmax=20)

fig, ax = plt.subplots(1, 3, figsize=(18, 6), subplot_kw={'projection': wcs})

# Adjust spacing to make room for the colorbar on the right
# (left, bottom, right, top, wspace, hspace)
plt.subplots_adjust(wspace=0.05, right=0.90) 

# Panel 1: NIR-Y
plot_map(y_mos, wcs=wcs, norm=shared_norm, colorbar=False, cmap='viridis', fig=fig, ax=ax[0])
ax[0].set_title('NIR-Y')
ax[0].grid(color='white', alpha=0.5)

# Panel 2: NIR-J
plot_map(j_mos, wcs=wcs, norm=shared_norm, colorbar=False, cmap='viridis', fig=fig, ax=ax[1])
ax[1].set_title('NIR-J')
ax[1].grid(color='white', alpha=0.5)
# Optional: Hide Y-axis labels for inner plots to save space
ax[1].coords['dec'].set_axislabel('')
ax[1].coords['dec'].set_ticklabel_visible(False)

# Panel 3: NIR-H
# We capture 'im' from the last plot to generate the colorbar
fig, ax[2], im = plot_map(h_mos, wcs=wcs, norm=shared_norm, colorbar=False, cmap='viridis', fig=fig, ax=ax[2])
ax[2].set_title('NIR-H')
ax[2].grid(color='white', alpha=0.5)
ax[2].coords['dec'].set_axislabel('')
ax[2].coords['dec'].set_ticklabel_visible(False)

# --- 3. Add the Single Colorbar ---

# Create a new axis for the colorbar [left, bottom, width, height]
# Coordinates are fractions of the figure size (0 to 1)
cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7]) 

cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label('$e^-$') # Label for the shared colorbar

plt.savefig('/home/thomasli/spherex/selfcal/figures/euclid_mosaics.png', dpi=200, bbox_inches='tight', transparent=False)
plt.close()